# Study report: stimulus-path

The study measures the hardware path of the stimulus (compare match, pin,
wire, input synchronizer, EXTI) with a software trigger (`NVIC->STIR`) in
the same image (`LATBENCH_SWTRIG`). It also explains why a wake that
preempts a time-sliced thread is slower than a wake from idle.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-measurement-validity@nucleo_g474re/st-stimulus-path"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## H1 and H2: edge and software trigger (irq-swtrig-1m)

`irq` starts at the edge, `irq_sw` at the STIR store, in the same image.
The clean hardware path is `irq` of `busy` minus `irq_sw`. The software
trigger never meets WFI, a kernel lock or the SysTick ISR, so `irq_sw`
has no tail (H2).

In [ ]:
if nbfig.built(inst, "irq-swtrig-1m"):
    df = inst.rows("irq-swtrig-1m")
    nbfig.dots(df, "Entry from the edge and from STIR (irq-swtrig-1m)", stats=("p50", "p99", "p99.99", "max")).show()
    display(nbfig.pivot(df, ["min", "p50", "p99", "p99.99", "max"]))

In [ ]:
if nbfig.built(inst, "irq-swtrig-1m"):
    nbfig.ccdf(nbfig.hists_of(inst, "irq-swtrig-1m"), "Entry from the edge and from STIR: CCDF", ["irq", "irq_sw"]).show()

## H3 and H4: the wake of a preempted thread (wake-swtrig-250k)

The p10 of `isr2thr` is in the fast mode, the p50 in the slow mode. In the
slow mode, the slice reset reprograms SysTick. If the preempted thread is
time-sliced, its slice timeout is aborted first, and the abort reprograms
SysTick once more (H3). `pr-config` is the configuration of the upstream
benchmark (H4).

In [ ]:
if nbfig.built(inst, "wake-swtrig-250k"):
    df = inst.rows("wake-swtrig-250k")
    nbfig.dots(df, "Wake from the edge and from STIR (wake-swtrig-250k)", stats=("min", "p50", "p99")).show()
    display(nbfig.pivot(df, ["min", "p50", "p99", "p99.99", "max"]))

In [ ]:
if nbfig.built(inst, "wake-swtrig-250k"):
    nbfig.ccdf(nbfig.hists_of(inst, "wake-swtrig-250k"), "ISR to thread: edge and STIR, CCDF", ["isr2thr", "isr2thr_sw"]).show()

In [ ]:
if nbfig.built(inst, "wake-swtrig-250k"):
    w = inst.tails("wake-swtrig-250k")
    display(inst.tail_causes("wake-swtrig-250k"))
    if len(w):
        nbfig.tails(w, "Witnesses of the largest wake samples (wake-swtrig-250k)").show()
        display(w.head(40))